### Imports

In [16]:
import pandas as pd
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
import pickle
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

### Read Data

In [2]:
train_df = pd.read_csv('./Data/train.csv')
test_df = pd.read_csv('./Data/test.csv')

### Describe Data

In [3]:
train_df.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

### Choose Data

In [5]:
# selecting only certain columns for practicing pipelines
# dropping any NaN values
select_df = train_df[[
    'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'LotShape', 
    'LandContour', 'Utilities', 'MiscVal', 'MoSold', 'YrSold', 'SaleType', 
    'SalePrice'
]].dropna()

In [6]:
X = pd.get_dummies(select_df.drop('SalePrice', axis=1))

y = select_df['SalePrice']

### Basic Pipeline Practice

In [7]:
# make_pipeline auto-names your steps
# pipeline = make_pipeline(StandardScaler(), RandomForestRegressor())

# Pipeline class makes you name your individual steps
pipeline = Pipeline([
    ('scaling', StandardScaler()),
    ('rfmodel', RandomForestRegressor())
])

In [14]:
pipeline

Pipeline(steps=[('scaling', StandardScaler()),
                ('rfmodel', RandomForestRegressor())])

In [8]:
pipeline.fit(X, y)

Pipeline(steps=[('scaling', StandardScaler()),
                ('rfmodel', RandomForestRegressor())])

In [9]:
pipeline.predict(X)

array([200520.5 , 164868.5 , 220777.  , ..., 224701.  , 147137.75,
       150146.  ])

### Save the Pipeline

In [10]:
with open('pipelinemodel.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

In [11]:
with open('pipelinemodel.pkl', 'rb') as f:
    pipeline_model = pickle.load(f)

In [12]:
pipeline_model.predict(X)

array([200520.5 , 164868.5 , 220777.  , ..., 224701.  , 147137.75,
       150146.  ])

In [13]:
# --- predicting with the reloaded model
# pipeline_model.predict(X)
# pipeline_model.steps[1][1].predict(X)

### Make the Full Pipeline Practice

ColumnTransformer can combine any number of different pipelines into one, large pipeline

In [28]:
# --- numerical features --- #
numerical_features = select_df.drop('SalePrice', axis=1).select_dtypes(
    exclude='object'
).columns

numerical_pipeline = Pipeline([('scaler', StandardScaler())])

# --- categorical features --- #
categorical_features = select_df.select_dtypes('object').columns

categorical_pipeline = Pipeline([('onehot', OneHotEncoder())])

In [29]:
# combining our numerical and categorical pipelines into one
column_transformer = ColumnTransformer([
    ('numeric_preprocessing', numerical_pipeline, numerical_features),
    ('categorical_preprocessing', categorical_pipeline, categorical_features)
])

column_transformer

ColumnTransformer(transformers=[('numeric_preprocessing',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 Index(['MSSubClass', 'LotFrontage', 'LotArea', 'MiscVal', 'MoSold', 'YrSold'], dtype='object')),
                                ('categorical_preprocessing',
                                 Pipeline(steps=[('onehot', OneHotEncoder())]),
                                 Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'SaleType'],
      dtype='object'))])

In [37]:
# combining our full preprocessing pipeline with a model
full_pipeline = Pipeline([
    ('preprocessing', column_transformer),
    ('randomforestregressor', RandomForestRegressor())
])

full_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric_preprocessing',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['MSSubClass', 'LotFrontage', 'LotArea', 'MiscVal', 'MoSold', 'YrSold'], dtype='object')),
                                                 ('categorical_preprocessing',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder())]),
                                                  Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'SaleType'],
      dtype='object'))])),
                ('randomforestregressor', RandomForestRegressor())])

In [31]:
X = select_df.drop('SalePrice', axis=1)
y = select_df['SalePrice']

In [32]:
full_pipeline.fit(X, y)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric_preprocessing',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['MSSubClass', 'LotFrontage', 'LotArea', 'MiscVal', 'MoSold', 'YrSold'], dtype='object')),
                                                 ('categorical_preprocessing',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder())]),
                                                  Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'SaleType'],
      dtype='object'))])),
                ('randomforestregressor', RandomForestRegressor())])

In [33]:
full_pipeline.predict(X)

array([202033.4       , 166963.71428571, 218557.        , ...,
       224602.75      , 145743.        , 152254.        ])

In [34]:
with open('full_pipeline.pkl', 'wb') as f:
    pickle.dump(full_pipeline, f)

In [35]:
with open('full_pipeline.pkl', 'rb') as f:
    model = pickle.load(f)

In [36]:
model

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric_preprocessing',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['MSSubClass', 'LotFrontage', 'LotArea', 'MiscVal', 'MoSold', 'YrSold'], dtype='object')),
                                                 ('categorical_preprocessing',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder())]),
                                                  Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'SaleType'],
      dtype='object'))])),
                ('randomforestregressor', RandomForestRegressor())])